### [ 생성 무게 예측 회귀 모델 구현 ]

- 데이터 관련 
    * 로딩 및 기본 확인 : EDA
    * 전처리 진행
    * 데이터 가공 (통계치 X) : 불필요 커럼 제거

- 학습 관련
    * 피쳐, 타겟 분리
    * 데이터셋 분리 : 학습용/검증용/테스트용
    * 학습용 데이터셋으로 가공 -> 데이터 누수 예방
    * 학습/검증/테스트 함수 

- 학습 진행
    * 학습 관련 설정값 <= 하이퍼파라미터들
    * 학습 관련 인스턴스들
    * 학습 진행 + 진행 과정에 대한 로그 : 학습과 검증의 loss, score 등등 
    
- 학습 평가
    * 학습 과정 로그기반 시각화 
    * 과대/과소/최적 평가

>> **[1] 데이터 준비 및 확인**

In [1]:
## --------------------------------------------------
## 모듈 로딩
## --------------------------------------------------
## 데이터 관련
import pandas as pd 
import numpy as np 

In [2]:
## --------------------------------------------------
## 데이터 선정 및 로딩
## --------------------------------------------------
## => 첫번째 줄 컬럼명 OK, 첫번째 컬럼 불필요 
fishDF = pd.read_csv('../../[8] 머신러닝/Data/Numbers/fish.csv', usecols=[1,2,3,4,5])

In [3]:
## --------------------------------------------------
## 데이터 기본 정보 확인 : 실제 데이터와 타입 체크 
## --------------------------------------------------
display(fishDF.head(2))
fishDF.info()

,Weight,Length,Diagonal,Height,Width
0,242.0,25.4,30.0,11.52,4.0200
1,290.0,26.3,31.2,12.48,4.3056


<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Weight    159 non-null    float64
 1   Length    159 non-null    float64
 2   Diagonal  159 non-null    float64
 3   Height    159 non-null    float64
 4   Width     159 non-null    float64
dtypes: float64(5)
memory usage: 6.3 KB


In [4]:
## --------------------------------------------------
## 컬럼별 값의 범위 확인 : 기술통계 확인
## --------------------------------------------------
fishDF.describe()

## [체크] --------------------------------------------
## 피쳐별 값의 범위 차이가 있음 => 스케일링 필요
## 타겟 컬럼 Weight의 최소값 0 => 결측치인데 0으로 채워진듯
##                             몇개 존재하는지 체크 필요
## --------------------------------------------------

,Weight,Length,Diagonal,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,28.415723,31.227044,8.970994,4.417486
std,357.978317,10.716328,11.610246,4.286208,1.685804
min,0.000000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,63.400000,68.000000,18.957000,8.142000


In [5]:
## -------------------------------------------
## 타겟 컬럼 Weight가 0인 행 즉, 샘플 개수 체크
## -------------------------------------------
zero_cnt = (fishDF['Weight'] == 0).sum()
zero_idx = fishDF[fishDF['Weight'] == 0].index.to_list()

print(f'무게가 0인 데이터 개수 : {zero_cnt}개, 행인덱스 번호 : {zero_idx}')


무게가 0인 데이터 개수 : 1개, 행인덱스 번호 : [40]


In [6]:
## 삭제 진행
cleanDF = fishDF.drop(index=zero_idx)
cleanDF.reset_index(drop=True, inplace=True)

In [7]:
## 확인 진행
cleanDF.head(), cleanDF.tail(), (cleanDF['Weight']==0).sum()

(   Weight  Length  Diagonal   Height   Width
 0   242.0    25.4      30.0  11.5200  4.0200
 1   290.0    26.3      31.2  12.4800  4.3056
 2   340.0    26.5      31.1  12.3778  4.6961
 3   363.0    29.0      33.5  12.7300  4.4555
 4   430.0    29.0      34.0  12.4440  5.1340,
      Weight  Length  Diagonal  Height   Width
 153    12.2    12.2      13.4  2.0904  1.3936
 154    13.4    12.4      13.5  2.4300  1.2690
 155    12.2    13.0      13.8  2.2770  1.2558
 156    19.7    14.3      15.2  2.8728  2.0672
 157    19.9    15.0      16.2  2.9322  1.8792,
 np.int64(0))

In [8]:
## 무게 0인 행 제거 후 다시 분포확인
cleanDF.describe()

## [확인] ----------------------------------------
## 타겟 Weight 컬럼의 값의 분포 : 넓게 퍼져있음. 
##                             최소-최대 차이 큼
##                             오른쪽으로 치우쳐 있음
## -----------------------------------------------

,Weight,Length,Diagonal,Height,Width
count,158.000000,158.000000,158.000000,158.000000,158.000000
mean,400.847468,28.465823,31.280380,8.986790,4.424232
std,357.697796,10.731707,11.627605,4.295191,1.689010
min,5.900000,8.400000,8.800000,1.728400,1.047600
25%,121.250000,21.000000,23.200000,5.940600,3.398650
50%,281.500000,27.400000,29.700000,7.789000,4.277050
75%,650.000000,35.750000,39.675000,12.371850,5.586750
max,1650.000000,63.400000,68.000000,18.957000,8.142000


>> **[2] 학습 데이터 준비**

In [9]:
## -----------------------------------------
## 모듈로딩
## -----------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [10]:
## -----------------------------------------
## [2-1] 피쳐와 타겟 분리
## -----------------------------------------
featureDF = cleanDF[cleanDF.columns[1:]]
targetSR  = cleanDF[cleanDF.columns[0]]

print(f'featureDF : {featureDF.shape} targetSR : {targetSR.shape}')

featureDF : (158, 4) targetSR : (158,)


In [11]:
## ----------------------------------------------
## [2-2] 타겟 컬럼 스케일링여부 결정
## ----------------------------------------------
targetSR.describe()

## [결정] ---------------------------------------
## - 값들의 범위가 넓게 퍼짐 즉, 값의 차이 큼!!
## - 오른쪽으로 즉, 큰 무게에 해당하는 데이터가 더 많음
## - 타겟도 스케일링 필요함
## - 값의 차이 크기 때문에 학습용 기준 스케일링 진행
## - 검증용/테스트용에 적용
## ----------------------------------------------

count     158.000000
mean      400.847468
std       357.697796
min         5.900000
25%       121.250000
50%       281.500000
75%       650.000000
max      1650.000000
Name: Weight, dtype: float64

In [12]:
## -----------------------------------------
## [2-3] 데이터셋 분리 : 학습용 | 검증용 | 테스트용
## -----------------------------------------
## => 학습용 : 테스트용 = 8 :2
X_train, X_test, Y_train, Y_test = train_test_split(featureDF,
                                                    targetSR,
                                                    test_size=0.2,
                                                    random_state=12)

## => 학습용 : 검증용   = 8 :2
X_train, X_valid, Y_train, Y_valid = train_test_split(  X_train,
                                                        Y_train,
                                                        test_size=0.2,
                                                        random_state=12)

In [13]:
## 데이터셋 확인
print(f'[TRAIN] {X_train.shape},  {Y_train.shape}')
print(f'[VALID] {X_valid.shape},  {Y_valid.shape}')
print(f'[TEST ] {X_test.shape},   {Y_test.shape}')

[TRAIN] (100, 4),  (100,)
[VALID] (26, 4),  (26,)
[TEST ] (32, 4),   (32,)


In [14]:
## -----------------------------------------
## [2-4] 학습용 기준 타겟 스케일링 : 데이터 누수
## - 딥러닝 학습 시 권장 사항
##   *안정적인 학습/자연스러운 기울기 업데이트 등의 이유로
##    평균0, 표준편차1이 되는 데이터를 좋아함
## -----------------------------------------
targetScaler = StandardScaler()

## 학습용 : SR -> DF : to_frame()
scY_train = targetScaler.fit_transform(Y_train.to_frame())

## 검증용, 테스트용 
scY_test  = targetScaler.transform(Y_test.to_frame())
scY_valid = targetScaler.transform(Y_valid.to_frame())


In [15]:
## 타겟 스케일링 후 확인
print(f'[TRAIN] {scY_train.shape},  {scY_train.dtype},  {scY_train.ndim}D')
print(f'[VALID] {scY_valid.shape},  {scY_valid.dtype},  {scY_valid.ndim}D')
print(f'[TEST ] {scY_test.shape},  {scY_test.dtype},  {scY_test.ndim}D')

[TRAIN] (100, 1),  float64,  2D
[VALID] (26, 1),  float64,  2D
[TEST ] (32, 1),  float64,  2D


In [16]:
## -----------------------------------------
## [2-5] 피쳐 스케일링 : By 학습용 스케일러
## -----------------------------------------
featureScaler = StandardScaler()

## 학습용 기준 스케일러 생성
scX_train = featureScaler.fit_transform(X_train)

## 검증용, 테스트용 
scX_test  = featureScaler.transform(X_test)
scX_valid = featureScaler.transform(X_valid)

In [17]:
## 타겟 스케일링 후 확인
print(f'[TRAIN] {scX_train.shape},  {scX_train.dtype},  {scX_train.ndim}D')
print(f'[VALID] {scX_valid.shape},  {scX_valid.dtype},  {scX_valid.ndim}D')
print(f'[TEST ] {scX_test.shape},  {scX_test.dtype},  {scX_test.ndim}D')

[TRAIN] (100, 4),  float64,  2D
[VALID] (26, 4),  float64,  2D
[TEST ] (32, 4),  float64,  2D


>> **[3] 학습 준비**

In [18]:
## -----------------------------------------
## 모듈로딩
## -----------------------------------------
## 학습관련 모듈들
import torch
from torch.nn import MSELoss                ## 손실함수    클래스
from torch.optim import Adam                ## 최적화     클래스
from torch.utils.data import DataLoader     ## 데이터로더  클래스

## 성능지표 관련 모듈
from sklearn.metrics import mean_squared_error, r2_score

## 사용자 정의 클래스 및 학습관련 함수 모듈
import sys
sys.path.append(r'D:\KDT\VS_KDT_14\[9]_DL\DAY05')
from dnn_fish_class import *
from reg_func import *


In [19]:
## ----------------------------------------------------------
## 학습 진행 관련 설정들
## ----------------------------------------------------------
## => 처음~끝까지 학습 횟수
EPOCHS = 50

## => 학습량 크기
BS = 25

## => W,B 업데이트/값 보정 간격
LR = 0.001

## => 학습 위치 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE => {DEVICE}')

DEVICE => cuda


In [20]:
## ----------------------------------------------------------
## 인스턴스 생성
## ----------------------------------------------------------
## => 모델 인스턴스 생성 + 저장 위치 설정
model = WeightRegression().to(DEVICE)

## => 최적화 인스턴스
adamOP = Adam(model.parameters(), lr=LR)

## => 손실함수 인스턴스 : 회귀용
lossFN = MSELoss()

## => 데이터로더 인스턴스
## => train : valid : test = 100 : 26 : 32
trainDL = DataLoader( WeightDataset(scX_train, scY_train), batch_size=BS, shuffle=True )
validDL = DataLoader( WeightDataset(scX_valid, scY_valid), batch_size=len(scX_valid) )
testDL  = DataLoader( WeightDataset(scX_test,  scY_test),  batch_size=len(scX_test) )


>> **[4] 학습 진행**

In [21]:
## ----------------------------------------------------------
## 학습 진행
## ----------------------------------------------------------
## 에포크 단위 손실과 정확도 저장 변수
epochHist = {'T_LOSS':[], 'T_R2':[], 'T_RMSE':[], 'V_LOSS':[],  'V_R2':[], 'V_RMSE':[]}

## 에포크 단위 학습/검증 진행 
for epo in range(EPOCHS):
    ## 학습
    t_loss, t_mse, t_rmse, t_r2 = training_regression( model, trainDL, 
                                                       lossFN, adamOP, DEVICE)
    ## 검증
    v_loss, v_mse, v_rmse, v_r2 = evaluating_regression( model, validDL, 
                                                         lossFN, DEVICE )
    
    ## 학습 및 검증 로그 저장 
    for key, value in zip(epochHist.keys(), [t_loss, t_r2, t_rmse, v_loss, v_r2, v_rmse]):
        epochHist[key].append(value)

    print(f"[EPOCH -{epo:03}]", end=' ')
    print(f"TRAIN  => LOSS : {t_loss:.6f},   MSE : {t_mse:.6f},   RMSE : {t_rmse:.6f},   R2 : {t_r2:.6f}", end=' ')
    print(f'VALID  => LOSS : {v_loss:.6f},   MSE : {v_mse:.6f},   RMSE : {v_rmse:.6f},   R2 : {v_r2:.6f}')


[EPOCH -000] TRAIN  => LOSS : 32.381099,   MSE : 1.295244,   RMSE : 1.138088,   R2 : -0.295244 VALID  => LOSS : 32.440934,   MSE : 1.247728,   RMSE : 1.117018,   R2 : -0.861161
[EPOCH -001] TRAIN  => LOSS : 31.790731,   MSE : 1.271629,   RMSE : 1.127665,   R2 : -0.271629 VALID  => LOSS : 31.412366,   MSE : 1.208168,   RMSE : 1.099167,   R2 : -0.802152
[EPOCH -002] TRAIN  => LOSS : 31.320824,   MSE : 1.252833,   RMSE : 1.119300,   R2 : -0.252833 VALID  => LOSS : 30.369590,   MSE : 1.168061,   RMSE : 1.080769,   R2 : -0.742327
[EPOCH -003] TRAIN  => LOSS : 30.773702,   MSE : 1.230948,   RMSE : 1.109481,   R2 : -0.230948 VALID  => LOSS : 29.367717,   MSE : 1.129528,   RMSE : 1.062792,   R2 : -0.684848
[EPOCH -004] TRAIN  => LOSS : 30.266763,   MSE : 1.210671,   RMSE : 1.100305,   R2 : -0.210671 VALID  => LOSS : 28.395524,   MSE : 1.092136,   RMSE : 1.045053,   R2 : -0.629073
[EPOCH -005] TRAIN  => LOSS : 29.732065,   MSE : 1.189283,   RMSE : 1.090542,   R2 : -0.189283 VALID  => LOSS : 27.

>> **[5] 학습평가** 

In [22]:
## --------------------------------------------------
## 학습과 검증 비교 시각화
## --------------------------------------------------

In [23]:
## --------------------------------------------------
## 테스트
## --------------------------------------------------
loss, mse, rmse, r2 = evaluating_regression( model, testDL, lossFN, DEVICE )

print(f'TEST 데이터셋에 대한 성능평가')
print(f'- Loss : {loss:.6f}\n- R2   : {r2:.6f}')
print(f'- MSE  : {mse:.6f}\n- RMSE : {rmse:.6f}')

TEST 데이터셋에 대한 성능평가
- Loss : 16.360245
- R2   : 0.528462
- MSE  : 0.511258
- RMSE : 0.715023


>> **[6] 모델 또는 파라미터 저장 & 부가적인 기능들 저장**

In [24]:
## -------------------------------------------
## [6-1] 모델 저장 경로 설정 및 파일명 
## -------------------------------------------
## 모듈 로딩
import os 

## 모델 저장 폴더
MODEL_DIR = '../Models'

## 모델 파일명
PARAMS_FILE = 'fish_model_weights.pth'     ## 층별 파라미터(W, b) 만 저장 
ALL_FILE    = 'fish_model_all.pt'          ## 모델 구조 + 층별 파라미터(W, b) 모두 저장


In [25]:
## 모델 폴더 체크 
os.makedirs(MODEL_DIR, exist_ok=True) 


## (1) 층별 파라미터(W, b) 만 저장 
torch.save(model.state_dict() , f'{MODEL_DIR}/{PARAMS_FILE}')


## (2) 모델 구조 +  층별 파라미터(W, b) 저장 
torch.save(model , f'{MODEL_DIR}/{ALL_FILE}')



In [26]:
## -------------------------------------------
## [6-2] 부가 기능 및 데이터 저장 
## -------------------------------------------
## 모듈 로딩
import joblib

## 저장 폴더
MODEL_DIR = '../Models'

## 전처리기 : 스케일러 2개
F_SCALER = 'featureScaler.pkl'      ## 입력 피쳐용 스케일러
T_SCALER = 'targetScaler.pkl'       ## 타겟용 스케일러

## 부가데이터 
F_COLS  =  'featureColos.pkl'       ## 입력 피쳐의 컬럼명 

In [27]:
## => 전처리기 : 스케일러 2개 저장
joblib.dump(featureScaler,  f'{MODEL_DIR}/{F_SCALER}')
joblib.dump(targetScaler,   f'{MODEL_DIR}/{T_SCALER}')

['../Models/targetScaler.pkl']

In [28]:
## => 부가데이터: 입력 피쳐의 컬럼명 저장 
joblib.dump(featureDF.columns.to_list(),  f'{MODEL_DIR}/{F_COLS}')


['../Models/featureColos.pkl']

>> **[7]예측**

In [29]:
## -------------------------------------------------------
## 새로운 생선의 무게 예측 
## -------------------------------------------------------
## -> 입력 : DataFrame으로 저장(DataFrame) -> 피쳐 스케일링(ndarray) -> Tensor화
## -> 출력 : Tensor -> 타겟 스케일로 복원된 2D 
## -------------------------------------------------------

## 정답 Weight = 242인 데이터라고 가정
## 피쳐 : Length, Diagonal, Height, Width
new_data = pd.DataFrame( [[25.4, 30, 11.52, 4.02]], columns=featureDF.columns )

## 학습용 데이터셋기반 피쳐 스케일러 사용 
scaled_data = featureScaler.transform(new_data)

## Tensor 변환
data = torch.tensor(scaled_data, dtype=torch.float32)

## 예측 : 타겟 스케일러로 복원된 예측값 반환
pred_weight = predict_regression( data, model, targetScaler, DEVICE)

print(f'예측 무게 : {pred_weight[0][0]}')

예측 무게 : 431.598876953125


In [30]:
## ----------------------------------------------------------
## [2] 저장된 모델 로딩해서 예측하기 : torch.load(모델파일)
## ----------------------------------------------------------
loadAModel =  torch.load(f'{MODEL_DIR}/{ALL_FILE}', weights_only=False)

## 예측 : 타겟 스케일러로 복원된 예측값 반환
pred_weight = predict_regression( data, loadAModel, targetScaler, DEVICE)

print(f'예측 무게 : {pred_weight[0][0]}')


예측 무게 : 431.598876953125


In [31]:
## ----------------------------------------------------------
## [3] 저장된 모델 파라미터 로딩해서 예측하기 : torch.load(모델파일)
## ----------------------------------------------------------
## 층별 가중치와 절편값만 존재
paramModel =  torch.load(f'{MODEL_DIR}/{PARAMS_FILE}', weights_only=True)

## 모델의 층별 가중치와 절편값으로 로딩
newModel = WeightRegression().to(DEVICE)           ## 모든 층의 가중치와 절편 초기화 
newModel.load_state_dict(paramModel)               ## 학습 완료 후 저장된 가중치와 절편 로딩

## 예측 : 타겟 스케일러로 복원된 예측값 반환
pred_weight = predict_regression( data, newModel, targetScaler, DEVICE)

print(f'예측 무게 : {pred_weight[0][0]}')

예측 무게 : 431.598876953125
